<a href="https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it: engineered features, categorical handling, fills.*

**The unit:** one row = one content page, described by its trailing-90-day numbers. **The slice:** the same pages as my baseline and model notebooks (500+ impressions, average position 1 to 20, at least 90 days old).

- **Engineered:** `log1p` of heavy-tailed counts (impressions, CTR, word count, search volume), and one flag, `is_new_to_search` (fewer than 30 days with impressions).
- **Categorical:** `content_type` and `main_intent` are one-hot encoded; a blank becomes its own `unknown` value.
- **Missing numbers:** median fill **plus a "was missing" flag column**, so a blank is never silently turned into a 0.
- **Never features:** the label, anything built from the last-30 and previous-30-day windows, and the identifiers.

(In the modeling notebooks the fills are learned inside each training fold. Here I fit on the whole slice only to show what the matrix looks like.)

In [1]:
import os, sys, subprocess, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Find the repo root so the CSV loads in Colab and locally.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yelsebety/Yassin-Flyrank.git"
REPO_DIR = "Yassin-Flyrank"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
raw["declined"] = (raw["trend_direction"] == "down").astype(int)   # outcome / evaluation only, never a feature

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

g = raw[(raw["impressions_90d"] >= 500) & (raw["avg_position"] > 0) & (raw["avg_position"] <= 20)
        & (raw["content_age_days"] >= 90)].copy().reset_index(drop=True)

g["log_impressions"]   = np.log1p(g["impressions_90d"])
g["log_ctr"]           = np.log1p(g["ctr"])
g["log_word_count"]    = np.log1p(g["word_count"])
g["log_search_volume"] = np.log1p(g["search_volume"])
g["is_new_to_search"]  = (g["days_with_impressions"] < 30).astype(int)

NUM = ["log_impressions", "avg_position", "log_ctr", "content_age_days", "days_since_last_update",
       "days_with_impressions", "is_new_to_search", "log_word_count", "log_search_volume",
       "competition", "cpc", "engagement_rate"]
CAT = ["content_type", "main_intent"]
FEATURES = NUM + CAT

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), NUM),
    ("cat", Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="unknown")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CAT),
])
X = pre.fit_transform(g[FEATURES])
names = list(pre.get_feature_names_out())
X = X.toarray() if hasattr(X, "toarray") else X

print(f"Pages: {len(g):,} | clients: {g['client_id'].nunique()} | one row = one page")
print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns "
      f"({len(NUM)} numeric + {len(CAT)} categorical inputs, plus 'was missing' flags and one-hot columns)")
assert not np.isnan(X).any(), "the matrix still has blanks"
print("No blanks left in the matrix; first 12 column names:", names[:12])

Pages: 12,023 | clients: 28 | one row = one page
Feature matrix: 12,023 rows x 24 columns (12 numeric + 2 categorical inputs, plus 'was missing' flags and one-hot columns)
No blanks left in the matrix; first 12 column names: ['num__log_impressions', 'num__avg_position', 'num__log_ctr', 'num__content_age_days', 'num__days_since_last_update', 'num__days_with_impressions', 'num__is_new_to_search', 'num__log_word_count', 'num__log_search_volume', 'num__competition', 'num__cpc', 'num__engagement_rate']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment I predict.*

**The decision moment:** the end of the 90-day snapshot, when a reviewer opens the queue. Every feature is a trailing-90-day measurement or a static page property, so it exists at that moment. **One honest catch:** the label ("down") compares the last 30 days with the 30 before, a window *inside* the 90 days my features cover. So features are not strictly earlier than the label; section 3 tests how much that matters.

**Before filling blanks, I check whether "missing" follows a category**, because a blind fill can quietly inject a category signal.

In [2]:
miss_cols = [c for c in ["word_count", "search_volume", "competition", "cpc", "engagement_rate"] if c in g.columns]
miss_by_type = g.groupby("content_type")[miss_cols].agg(lambda s: s.isna().mean()).round(3)
miss_by_type["pages"] = g.groupby("content_type").size()
print("Share missing, by content type (n = pages in that type):")
display(miss_by_type)

notes = [
    ("log_impressions", "log of search impressions, 90 days", "numeric", "trailing 90 days"),
    ("avg_position", "average search position (1 = top); 0 would mean no data", "numeric", "trailing 90 days"),
    ("log_ctr", "log of click-through rate (a % value)", "numeric", "trailing 90 days"),
    ("content_age_days", "days since the page was published", "numeric", "static at the snapshot"),
    ("days_since_last_update", "days since the last content update", "numeric", "static at the snapshot"),
    ("days_with_impressions", "days (of 90) with at least one impression", "numeric", "trailing 90 days"),
    ("is_new_to_search", "1 if fewer than 30 days with impressions", "numeric flag", "derived from the line above"),
    ("log_word_count", "log of word count", "numeric", "static at the snapshot"),
    ("log_search_volume", "log of the keyword's search volume", "numeric", "keyword data, external"),
    ("competition", "keyword competition (0 to 1)", "numeric", "keyword data, external"),
    ("cpc", "keyword cost per click", "numeric", "keyword data, external"),
    ("engagement_rate", "share of engaged analytics sessions", "numeric", "trailing 90 days, needs analytics"),
    ("content_type", "kind of content", "categorical (one-hot)", "static at the snapshot"),
    ("main_intent", "dominant search intent of the page's keyword", "categorical (one-hot)", "keyword data, external"),
]
base_of = {"log_impressions": "impressions_90d", "log_ctr": "ctr", "log_word_count": "word_count", "log_search_volume": "search_volume"}
rows = []
for f, meaning, kind, when in notes:
    src = base_of.get(f, f)
    miss = g[src].isna().mean() if src in g.columns else float("nan")
    handling = ("blank becomes 'unknown'" if kind.startswith("categorical")
                else ("median fill + 'was missing' flag" if miss > 0 else "no blanks"))
    rows.append({"feature": f, "meaning": meaning, "kind": kind, "missing": round(miss, 3),
                 "missing handling": handling, "available when?": when})
feature_notes = pd.DataFrame(rows)
display(feature_notes)

Share missing, by content type (n = pages in that type):


,word_count,search_volume,competition,cpc,engagement_rate,pages
content_type,,,,,,
comparison article,0.000,0.000,0.000,0.000,0.0,61
feedly article,0.000,1.000,1.000,1.000,0.0,120
keyword article,0.309,0.009,0.009,0.009,0.0,11842


,feature,meaning,kind,missing,missing handling,available when?
0,log_impressions,"log of search impressions, 90 days",numeric,0.000,no blanks,trailing 90 days
1,avg_position,average search position (1 = top); 0 would mean no data,numeric,0.000,no blanks,trailing 90 days
2,log_ctr,log of click-through rate (a % value),numeric,0.000,no blanks,trailing 90 days
3,content_age_days,days since the page was published,numeric,0.000,no blanks,static at the snapshot
4,days_since_last_update,days since the last content update,numeric,0.000,no blanks,static at the snapshot
5,days_with_impressions,days (of 90) with at least one impression,numeric,0.000,no blanks,trailing 90 days
6,is_new_to_search,1 if fewer than 30 days with impressions,numeric flag,0.000,no blanks,derived from the line above
7,log_word_count,log of word count,numeric,0.305,median fill + 'was missing' flag,static at the snapshot
8,log_search_volume,log of the keyword's search volume,numeric,0.019,median fill + 'was missing' flag,"keyword data, external"
9,competition,keyword competition (0 to 1),numeric,0.019,median fill + 'was missing' flag,"keyword data, external"


## 3. The leakage hunt

*Attack my own features: label-derived columns, future windows, product flags. Show the test.*

1. **Name scan:** which columns look like labels, trends, flags or scores, and are any in my features?
2. **Too-good-to-be-true scan:** each feature's single-feature strength against the outcome. Above about 0.75 AUC is a leakage smell. I also scan suspicious columns I did *not* use, to show the scan can find a leak.
3. **Deliberate leak:** add one label-derived column, watch the score jump, then remove it.
4. **Strictly-earlier window test:** the same model using only the first 30 days of the snapshot.
5. **Identifier check:** IDs are for grouping, never features.
6. **Privacy scan:** no URLs, domains or e-mail addresses in text columns.

In [3]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# ---- 1. Name scan ----
TOKENS = {"trend": "label-derived (built from the outcome windows)", "last_30": "inside the label window",
          "prev_30": "inside the label window", "declin": "label-derived", "health": "product flag / score",
          "score": "product flag / score", "flag": "product flag", "quick": "product flag", "recommend": "product decision",
          "priority": "product decision", "action": "product decision", "status": "product decision",
          "tier": "bucketed copy of another feature", "level": "bucketed copy of another feature"}
hits = []
for c in raw.columns:
    for tok, why in TOKENS.items():
        if tok in c.lower():
            hits.append({"column": c, "matched": tok, "why it is a risk": why, "used as a feature?": c in FEATURES})
            break
print("1. Columns whose names look like labels, trends, flags or bucketed copies:")
display(pd.DataFrame(hits))
assert not any(h["used as a feature?"] for h in hits), "a risky column is in the feature list"
print("   PASS: none of them is a feature.\n")

# ---- 2. Single-feature strength ----
def single_auc(s, y):
    s = s.fillna(s.median())
    a = roc_auc_score(y, s)
    return max(a, 1 - a)
checks = [c for c in NUM if c in g.columns]
suspects = [c for c in ["trend_pct", "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d"] if c in g.columns]
scan = pd.DataFrame([{"column": c, "role": "my feature", "single-feature AUC": single_auc(g[c], g["declined"])} for c in checks]
                    + [{"column": c, "role": "NOT used (suspect)", "single-feature AUC": single_auc(g[c], g["declined"])} for c in suspects])
scan["leakage smell (>= 0.75)"] = scan["single-feature AUC"] >= 0.75
print("2. Too-good-to-be-true scan:")
display(scan.sort_values("single-feature AUC", ascending=False).round(3))
assert not scan[scan["role"] == "my feature"]["leakage smell (>= 0.75)"].any(), "a feature of mine is too good to be true"
print("   PASS: none of my features trips the alarm; the suspect columns show what an alarm looks like.\n")

1. Columns whose names look like labels, trends, flags or bucketed copies:


,column,matched,why it is a risk,used as a feature?
0,competition_level,level,bucketed copy of another feature,False
1,impressions_last_30d,last_30,inside the label window,False
2,clicks_last_30d,last_30,inside the label window,False
3,sessions_last_30d,last_30,inside the label window,False
4,impressions_prev_30d,prev_30,inside the label window,False
5,clicks_prev_30d,prev_30,inside the label window,False
6,sessions_prev_30d,prev_30,inside the label window,False
7,age_tier,tier,bucketed copy of another feature,False
8,age_tier_order,tier,bucketed copy of another feature,False
9,freshness_tier,tier,bucketed copy of another feature,False


   PASS: none of them is a feature.

2. Too-good-to-be-true scan:


,column,role,single-feature AUC,leakage smell (>= 0.75)
12,trend_pct,NOT used (suspect),0.997,True
13,impressions_last_30d,NOT used (suspect),0.723,False
15,clicks_last_30d,NOT used (suspect),0.672,False
2,log_ctr,my feature,0.598,False
16,clicks_prev_30d,NOT used (suspect),0.597,False
0,log_impressions,my feature,0.591,False
3,content_age_days,my feature,0.572,False
11,engagement_rate,my feature,0.562,False
5,days_with_impressions,my feature,0.561,False
14,impressions_prev_30d,NOT used (suspect),0.547,False


   PASS: none of my features trips the alarm; the suspect columns show what an alarm looks like.



In [4]:
# ---- 3. Deliberate leak, and 4. the first-30-day window test (grouped by client) ----
def make_model(name, num_cols):
    steps = [("fill", SimpleImputer(strategy="median", add_indicator=True))]
    if name == "Logistic Regression":
        steps.append(("scale", StandardScaler()))
    prep = ColumnTransformer([("num", Pipeline(steps), num_cols),
                              ("cat", Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="unknown")),
                                                ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CAT)])
    est = (LogisticRegression(C=1.0, max_iter=2000) if name == "Logistic Regression"
           else RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=0))
    return Pipeline([("pre", prep), ("m", est)])

def grouped_auc(df, name, num_cols):
    aucs = []
    for tr, va in GroupKFold(n_splits=5).split(df, df["declined"], df["client_id"]):
        a, b = df.iloc[tr], df.iloc[va]
        m = make_model(name, num_cols).fit(a[num_cols + CAT], a["declined"])
        aucs.append(roc_auc_score(b["declined"], m.predict_proba(b[num_cols + CAT])[:, 1]))
    return float(np.mean(aucs))

rows = []
for name in ("Logistic Regression", "Random Forest"):
    rows.append({"model": name, "AUC, honest features": grouped_auc(g, name, NUM),
                 "AUC, with trend_pct added": grouped_auc(g, name, NUM + ["trend_pct"])})
leak_test = pd.DataFrame(rows).round(3)
print("3. Deliberate leak (trend_pct is built from the same windows as the label), grouped by client:")
display(leak_test)
assert (leak_test["AUC, with trend_pct added"] > leak_test["AUC, honest features"] + 0.2).all(), "the leak did not show up, so the harness is suspect"
print("   PASS: the planted leak jumps the score, so the harness can see leakage. The honest column is the one I keep.\n")

g["impr_first30"] = g["impressions_90d"] - g["impressions_last_30d"] - g["impressions_prev_30d"]
g["clk_first30"] = g["clicks_90d"] - g["clicks_last_30d"] - g["clicks_prev_30d"]
print(f"4. Rows where first-30-day impressions come out negative (would break the assumption): {(g['impr_first30'] < 0).sum()}")
g["log_impr_first30"] = np.log1p(g["impr_first30"].clip(lower=0))
g["log_ctr_first30"] = np.log1p(np.where(g["impr_first30"] > 0, 100 * g["clk_first30"] / g["impr_first30"], np.nan))
NUM_EARLY = ["log_impr_first30", "log_ctr_first30", "content_age_days", "days_since_last_update",
             "log_word_count", "log_search_volume", "competition", "cpc"]
early = pd.DataFrame([{"model": "Logistic Regression", "AUC, all 90-day features": grouped_auc(g, "Logistic Regression", NUM),
                       "AUC, first-30-day features only": grouped_auc(g, "Logistic Regression", NUM_EARLY)}]).round(3)
display(early)
print("   The early-window version also drops features with no first-30-day form (position, active days, engagement),")
print("   so a lower score mixes the overlap effect with those losses. I cannot isolate the overlap with this data.\n")

3. Deliberate leak (trend_pct is built from the same windows as the label), grouped by client:


,model,"AUC, honest features","AUC, with trend_pct added"
0,Logistic Regression,0.628,0.998
1,Random Forest,0.615,1.000


   PASS: the planted leak jumps the score, so the harness can see leakage. The honest column is the one I keep.

4. Rows where first-30-day impressions come out negative (would break the assumption): 0


,model,"AUC, all 90-day features","AUC, first-30-day features only"
0,Logistic Regression,0.628,0.593


   The early-window version also drops features with no first-30-day form (position, active days, engagement),
   so a lower score mixes the overlap effect with those losses. I cannot isolate the overlap with this data.



In [5]:
# ---- 5. Identifier check ----
ID_COLS = [c for c in ["client_id", "content_id"] if c in raw.columns]
assert not set(ID_COLS) & set(FEATURES), "an identifier is a feature"
print("5. PASS: identifiers", ID_COLS, "are used for grouping only (GroupKFold on client_id), never as features.")

# ---- 6. Privacy scan ----
patterns = {"url": r"https?://|www\.", "e-mail": r"[\w.+-]+@[\w-]+\.[\w.]+",
            "domain-like": r"\b[a-z0-9-]{2,}\.(?:com|net|org|io|co|ai|info|biz|edu|gov|eg|uk|de)\b"}
text_cols = [c for c in raw.columns if raw[c].dtype == object]
found = []
for c in text_cols:
    s = raw[c].dropna().astype(str).str.lower()
    for label, pat in patterns.items():
        n = int(s.str.contains(pat, regex=True).sum())
        if n:
            found.append({"column": c, "pattern": label, "rows": n})
print(f"6. Privacy scan over {len(text_cols)} text columns ({', '.join(text_cols)}):")
print("   PASS: no URL, domain or e-mail patterns found." if not found else "   FOUND:")
if found:
    display(pd.DataFrame(found))

5. PASS: identifiers ['client_id', 'content_id'] are used for grouping only (GroupKFold on client_id), never as features.
6. Privacy scan over 14 text columns (content_id, client_id, competition_level, content_type, main_intent, provider_used, model_used, age_tier, freshness_tier, word_count_tier, char_count_tier, impression_tier, position_tier, trend_direction):
   PASS: no URL, domain or e-mail patterns found.


## 4. What I excluded and why

*The fields I refused to use, with one line of why each.* The table is built from the real column list, so it only names fields that exist in the data.

In [6]:
excluded_reasons = {
    "trend_direction": "the outcome label itself",
    "trend_pct": "the outcome as a number: built from the last-30 vs. previous-30-day windows (the deliberate leak above)",
    "impressions_last_30d": "inside the label window: the label is computed from it",
    "impressions_prev_30d": "inside the label window: the label is computed from it",
    "clicks_last_30d": "inside the label window",
    "clicks_prev_30d": "inside the label window",
    "sessions_last_30d": "inside the label window",
    "sessions_prev_30d": "inside the label window",
    "clicks_90d": "redundant with CTR and impressions (CTR = clicks / impressions)",
    "client_id": "identifier: used only to keep a client's pages together when splitting",
    "content_id": "identifier: one value per page, would let a model memorise pages",
    "position_tier": "a bucketed copy of avg_position",
    "freshness_tier": "a bucketed copy of days_since_last_update",
    "competition_level": "a bucketed copy of competition",
}
present = {k: v for k, v in excluded_reasons.items() if k in raw.columns}
for c in [h["column"] for h in hits if h["column"] not in present and h["column"] not in FEATURES]:
    present[c] = "name looks like a label, trend, flag or score; not used (check what it is before ever using it)"
excluded = pd.DataFrame({"excluded field": list(present), "why": list(present.values())})
assert not set(excluded["excluded field"]) & set(FEATURES), "an excluded field is also a feature"
print(f"{len(excluded)} excluded fields (all present in the data); none is a feature.")
display(excluded)
print("Product flags and health scores, if they ever appear in a release, are OUTPUTS to beat, never inputs: using them would copy a decision already made.")

20 excluded fields (all present in the data); none is a feature.


,excluded field,why
0,trend_direction,the outcome label itself
1,trend_pct,the outcome as a number: built from the last-30 vs. previous-30-day windows (the deliberate leak above)
2,impressions_last_30d,inside the label window: the label is computed from it
3,impressions_prev_30d,inside the label window: the label is computed from it
4,clicks_last_30d,inside the label window
5,clicks_prev_30d,inside the label window
6,sessions_last_30d,inside the label window
7,sessions_prev_30d,inside the label window
8,clicks_90d,redundant with CTR and impressions (CTR = clicks / impressions)
9,client_id,identifier: used only to keep a client's pages together when splitting


Product flags and health scores, if they ever appear in a release, are OUTPUTS to beat, never inputs: using them would copy a decision already made.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.